In [ ]:
import polars as pl
from datasets import load_dataset

In [ ]:
markets = pl.read_parquet('data/processed/markets_political.parquet')

In [ ]:
for i,m in enumerate(markets.iter_rows(named=True)):
    print(i,m['condition_id'])

In [ ]:
desired_markets = markets['condition_id'].to_list()

In [ ]:
# Stream just the first batch to check schema and content
dataset = load_dataset(
    "SII-WANGZJ/Polymarket_data",
    data_files="quant.parquet",
    streaming=True,
    split="train",
    filters=[("condition_id", "in", desired_markets)]
)

In [ ]:
rows = []
for i,x in enumerate(dataset):
    rows.append(x)
    if i % 10000 == 0:
        print(f"Loaded {i} rows")

    if i == 999_000:
        break
        

In [ ]:
sample = pl.DataFrame(rows)

In [ ]:
for s in sample.group_by('condition_id').agg([
    pl.len().alias('count')
]).sort('count')[-1].iter_rows(named=True):
    print(s)

In [22]:
import polars as pl
from datetime import timedelta

sample = pl.read_parquet('data/sample/quant_sample_political.parquet')
markets = pl.read_parquet('data/processed/markets_political.parquet')

# Find which market has the most trades in your sample
best_markets = (sample
    .group_by("condition_id")
    .len()
    .sort("len", descending=True)
    .head(10))

print("Markets with most trades in sample:")
print(best_markets)

# Use the top market
c_id = best_markets["condition_id"][0]
print(f"\nUsing condition_id: {c_id}")

# Convert timestamp to datetime
sample = sample.with_columns(
    pl.from_epoch(pl.col('timestamp'), time_unit='s').alias('datetime')
).with_columns(
    pl.col('datetime').dt.convert_time_zone('UTC').dt.cast_time_unit('ms')
)

# Filter to single market
m_sample = sample.filter(pl.col('condition_id') == c_id)

# Use latest available trade as the reference point instead of end_date
# This is only for testing on the sample — on full data you use end_date
latest_trade = m_sample['datetime'].max()
window_start = latest_trade - timedelta(days=30)

window_trades = m_sample.filter(
    (pl.col('datetime') >= window_start) &
    (pl.col('datetime') <= latest_trade)
)

print(f"Latest trade: {latest_trade}")
print(f"Window start: {window_start}")
print(f"Trades in window: {len(window_trades)}")
print(window_trades.select(['datetime', 'price', 'usd_amount', 'side']))

Markets with most trades in sample:
shape: (10, 2)
┌─────────────────────────────────┬───────┐
│ condition_id                    ┆ len   │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ 0xdd22472e552920b8438158ea7238… ┆ 62946 │
│ 0xc6485bb7ea46d7bb89beb9c91e75… ┆ 31229 │
│ 0x653483009043f1663360bc35ed8f… ┆ 29151 │
│ 0x9e9071636d176562592a98dfede8… ┆ 26848 │
│ 0x3ac961dca3c4f0fc34bf94c66196… ┆ 24631 │
│ 0x14018049e265a2d88f284be9588e… ┆ 23954 │
│ 0x230144e34a84dfd0ebdc6de7fde3… ┆ 20263 │
│ 0x7da35195ac3c7bf167f88ab0c270… ┆ 20086 │
│ 0xd6f4c41fd20b8c6d5b48d528ead9… ┆ 19105 │
│ 0x40bbdd26dc08406eedcb913efee7… ┆ 18786 │
└─────────────────────────────────┴───────┘

Using condition_id: 0xdd22472e552920b8438158ea7238bfadfa4f736aa4cee91a6b86c39ead110917
Latest trade: 2024-07-31 23:53:54+00:00
Window start: 2024-07-01 23:53:54+00:00
Trades in window: 40483
shape: (40_483, 4)
┌─────────────────────────┬───────

In [25]:
price_series = window_trades['price']

In [26]:
# Feature engineering

import numpy as np

# Price features
price_start = price_series.first()  
price_end = price_series.last()
price_mean = price_series.mean()
price_min = price_series.min()
price_max = price_series.max()
price_volatility = price_series.std()
price_range = price_max - price_min
price_momentum = price_end - price_start

# Volume and activity features
log_total_volume = np.log1p(window_trades['usd_amount'].sum())
log_trade_amount = np.log1p(len(window_trades))
log_avg_trade_size = np.log1p(window_trades['usd_amount'].mean())

# Sentiment feature
buy_ratio = (window_trades['side'] == 'BUY').mean()

In [28]:
print(f"price_start:      {price_start:.4f}")
print(f"price_end:        {price_end:.4f}")
print(f"price_mean:       {price_mean:.4f}")
print(f"price_min:        {price_min:.4f}")
print(f"price_max:        {price_max:.4f}")
print(f"price_volatility: {price_volatility:.4f}")
print(f"price_range:      {price_range:.4f}")
print(f"price_momentum:   {price_momentum:.4f}")
print(f"log_total_volume:   {log_total_volume:.4f}")
print(f"log_trade_count:    {log_trade_amount:.4f}")
print(f"log_avg_trade_size: {log_avg_trade_size:.4f}")
print(f"buy_ratio:          {buy_ratio:.4f}")

price_start:      0.6900
price_end:        0.6900
price_mean:       0.6293
price_min:        0.5500
price_max:        0.7500
price_volatility: 0.0356
price_range:      0.2000
price_momentum:   0.0000
log_total_volume:   16.6490
log_trade_count:    10.6087
log_avg_trade_size: 6.0428
buy_ratio:          0.4729


In [14]:
window_start

end_date
"datetime[ms, UTC]"
2024-10-06 12:00:00 UTC


In [16]:
window_trades

timestamp,block_number,transaction_hash,log_index,market_id,condition_id,event_id,price,usd_amount,token_amount,side,maker,taker,datetime
i64,i64,str,i64,str,str,str,f64,f64,f64,str,str,str,"datetime[ms, UTC]"


In [ ]:
window_trades

In [ ]:
sample_markets = sample['condition_id'].unique().to_list()

In [ ]:
duplicates = list(set(sample_markets) & set(desired_markets))

In [ ]:
len(duplicates)

In [ ]:
sample.sort('timestamp', descending=False)

In [ ]:
sample.write_parquet('data/sample/quant_sample_political.parquet')